# 🎙️ speakerscribe — Speech-to-Text with Speaker Diarization

**Transcribe audio/video files with automatic speaker identification.**

Powered by [faster-whisper](https://github.com/SYSTRAN/faster-whisper) + [pyannote.audio](https://github.com/pyannote/pyannote-audio).

---

## ✅ Before you start — checklist

1. **GPU runtime**: `Runtime → Change runtime type → T4 GPU`
2. **HuggingFace token** (required for speaker diarization):
   - Create a **Read** token at https://huggingface.co/settings/tokens
   - Accept the model terms at https://huggingface.co/pyannote/speaker-diarization-community-1
   - Add it to **Colab Secrets** (🔑 icon in the left sidebar) with the name `HF_TOKEN`
3. **Place your audio/video files** in your Google Drive, inside a folder called `data/` within your project workspace.

---

> 💡 **Supported formats:** `.mp4` `.mp3` `.wav` `.m4a` `.mkv` `.aac` `.flac` `.ogg` `.webm`

## Step 1 — Install speakerscribe

> ⚠️ After the cell finishes, **restart the runtime** (`Runtime → Restart runtime`) and then run all remaining cells.

In [ ]:
# Install speakerscribe and its dependencies
# This takes 3-5 minutes on the first run (downloading model weights separately)
%pip install speakerscribe --quiet

print("✅ Installation complete.")
print("⚠️  Please restart the runtime now: Runtime → Restart runtime")
print("   Then run all remaining cells (do NOT re-run this cell).")

## Step 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
print("✅ Google Drive mounted at /content/drive")

## Step 3 — Configuration

**Only edit the cells in this section.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  ✏️  EDIT HERE — all other cells can run without changes
# ══════════════════════════════════════════════════════════════════

# Path to your project workspace in Google Drive.
# Place your audio/video files inside <WORKSPACE>/data/
WORKSPACE = "/content/drive/MyDrive/speakerscribe_project"

# Whisper model. Options (GPU VRAM required):
#   'tiny'           (~1.5 GB)  — fastest, basic quality
#   'small'          (~2.5 GB)  — fast, decent quality
#   'large-v3-turbo' (~3.5 GB)  — recommended: fast + excellent quality ⭐
#   'large-v3'       (~5.0 GB)  — best quality, slower
MODEL = "large-v3-turbo"

# Language code (None = auto-detect). Examples: 'en', 'es', 'fr', 'pt'
LANGUAGE = None

# Speaker diarization (requires HF_TOKEN in Colab Secrets)
ENABLE_DIARIZATION = True

# Optional: pin the exact number of speakers if you know it.
# Leave as None if you don't know in advance.
NUM_SPEAKERS = None   # e.g. 2 for a one-on-one interview

# Glossary / initial prompt: helps with proper nouns, brand names, jargon.
# Leave empty string for no prompt.
INITIAL_PROMPT = ""

# Force re-processing of files even if outputs already exist.
FORCE_REPROCESS = False

# ══════════════════════════════════════════════════════════════════
print(f"Workspace : {WORKSPACE}")
print(f"Model     : {MODEL}")
print(f"Language  : {LANGUAGE or 'auto-detect'}")
print(f"Diarize   : {ENABLE_DIARIZATION}")
print(f"Speakers  : {NUM_SPEAKERS or 'auto-detect'}")

## Step 4 — Pre-flight check

Validates GPU, disk space, HF token, and detects your audio files — before loading any models.

In [ ]:
import speakerscribe
from speakerscribe import TranscriptionConfig, WorkspacePaths
from speakerscribe.pipeline import preflight_check

print(f"speakerscribe version: {speakerscribe.__version__}")

config = TranscriptionConfig(
    model=MODEL,
    language=LANGUAGE,
    beam_size=5,
    enable_diarization=ENABLE_DIARIZATION,
    num_speakers=NUM_SPEAKERS,
    initial_prompt=INITIAL_PROMPT or None,
    force_reprocess=FORCE_REPROCESS,
    evaluate_quality=True,
)

paths = WorkspacePaths(workspace=WORKSPACE)
paths.create_directories()

# Show what files are in data/
media = paths.list_media_files()
print(f"\n📂 Files found in {paths.data}:")
for i, f in enumerate(media, 1):
    size_mb = f.stat().st_size / 1e6
    print(f"   {i}. {f.name}  ({size_mb:.1f} MB)")

if not media:
    print("\n⚠️  No audio files found!")
    print(f"   Place your files inside: {paths.data}")
else:
    # Run pre-flight checks
    info = preflight_check(paths, config)
    print(f"\n✅ Pre-flight OK — {info['n_files']} file(s) ready to process")

## Step 5 — Transcribe

Processes all files in `<WORKSPACE>/data/`. The Whisper model is loaded once and reused for all files.

In [ ]:
from speakerscribe import process_batch

results = process_batch(paths, config)

# Summary
n_ok    = sum(1 for r in results if r.get("status") == "ok")
n_skip  = sum(1 for r in results if r.get("status") == "skipped")
n_err   = sum(1 for r in results if r.get("status") == "error")
n_words = sum(r.get("total_words", 0) for r in results if r.get("status") == "ok")

print(f"\n🎉 Done!  OK={n_ok}  Skipped={n_skip}  Errors={n_err}  Total words={n_words:,}")
print(f"\n📁 Outputs saved to: {paths.transcripts}")
print(f"📁 Splits saved to  : {paths.splits}")

## Step 6 (optional) — Inspect a transcription

In [ ]:
from speakerscribe import inspect_json
from pathlib import Path

# Change this to the actual JSON file you want to inspect
JSON_FILE = str(paths.transcripts) + "/YOUR_FILE_large-v3-turbo.json"

if Path(JSON_FILE).exists():
    info = inspect_json(Path(JSON_FILE))
else:
    # List available files
    json_files = list(paths.transcripts.glob("*.json"))
    if json_files:
        print("Available JSON files:")
        for f in json_files:
            print(f"  {f.name}")
        # Auto-inspect the first one
        print("\n--- Inspecting first file ---")
        info = inspect_json(json_files[0])
    else:
        print("No JSON files found yet. Run Step 5 first.")

## Step 7 (optional) — Rename speaker labels

After reading the transcript and identifying who is who, replace `SPEAKER_00`, `SPEAKER_01`, etc. with real names.

In [ ]:
from speakerscribe import rename_speakers_in_outputs

# ✏️ Edit these values:
BASE_NAME = "your_audio_file_large-v3-turbo"  # filename stem + model name (no extension)
SPEAKER_MAPPING = {
    "SPEAKER_00": "Alice",
    "SPEAKER_01": "Bob",
    # Add more speakers as needed
}

stats = rename_speakers_in_outputs(paths, BASE_NAME, SPEAKER_MAPPING)
print(f"\n✅ Renamed speakers in {len(stats)} file(s)")
for filename, count in stats.items():
    print(f"   {filename}: {count} replacements")

## Step 8 (optional) — Clean outputs

Remove outputs for a specific file so you can re-process it with different settings.

In [ ]:
from speakerscribe import delete_outputs_for

# ✏️ Set a pattern that matches the filename(s) you want to remove
PATTERN = "your_audio_file"  # deletes all outputs that contain this substring

n_deleted = delete_outputs_for(paths, pattern=PATTERN, include_diar_cache=True)
print(f"✅ {n_deleted} file(s) deleted — ready to re-process")

---

## 📖 Tips

- **Colab session reset?** Re-run Steps 2–5. Outputs on Drive are safe.
- **Already processed?** speakerscribe skips files with existing outputs unless `FORCE_REPROCESS = True`.
- **Diarization cache** is saved in `_diar_cache/`. If you re-run with the same audio, pyannote is not called again.
- **Splits** in `splits/` are ready to paste into ChatGPT, Claude, or any other LLM for summarization.
- **Quality warnings** in the log? Check the `REPETITIONS` flag — if present, try reducing `beam_size` or adding an `initial_prompt`.

---

📦 [speakerscribe on PyPI](https://pypi.org/project/speakerscribe/) · 
📁 [GitHub](https://github.com/EnriqueForero/speakerscribe) · 
🐛 [Report an issue](https://github.com/EnriqueForero/speakerscribe/issues)